# 🛡️ PhishShield - Model 1: Classical NLP (TF-IDF + Calibrated Classifier)

This notebook trains, evaluates, and exports the **Classical NLP Model** for phishing email detection using:
- **Dataset:** `phishing_email.csv` (~82,486 emails)
- **Feature Extraction:** Sublinear TF-IDF with Unigrams and Bigrams (`max_features=25,000`)
- **Model:** Calibrated Logistic Regression
- **Target:** Predict probability of phishing (0 = Legitimate, 1 = Phishing)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve

print("Libraries successfully imported!")

## 1. Load and Inspect Dataset

In [ ]:
df = pd.read_csv("phishing_email.csv")
print(f"Total rows: {len(df)}")
print("Columns:", df.columns.tolist())
print("\nClass balance:")
print(df["label"].value_counts())
df.head(5)

## 2. Train / Test Split (Stratified 80/20)

In [ ]:
X = df["text_combined"].astype(str)
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {len(X_train)} samples")
print(f"Testing set:  {len(X_test)} samples")

## 3. Build & Train the TF-IDF + Logistic Regression Pipeline

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=25000,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        C=2.0,
        random_state=42
    ))
])

print("Fitting pipeline on training data...")
pipeline.fit(X_train, y_train)
print("Training complete!")

## 4. Evaluation Metrics

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {acc * 100:.2f}%")
print(f"ROC-AUC  : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate (0)", "Phishing (1)"], digits=4))

## 5. Confusion Matrix & ROC Curve Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Legit", "Phishing"], yticklabels=["Legit", "Phishing"], ax=axes[0])
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="crimson", lw=2, label=f"ROC curve (AUC = {auc:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Receiver Operating Characteristic (ROC)")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Top Phishing Indicators (Model Interpretability)

In [ ]:
feature_names = np.array(pipeline.named_steps["tfidf"].get_feature_names_out())
coefs = pipeline.named_steps["clf"].coef_[0]

top_phish_idx = np.argsort(coefs)[-15:]
top_legit_idx = np.argsort(coefs)[:15]

print("🚨 Top 15 Phishing Keywords/Phrases:")
for word, weight in zip(feature_names[top_phish_idx][::-1], coefs[top_phish_idx][::-1]):
    print(f"  • {word:<25} (weight: {weight:+.2f})")

print("\n✅ Top 15 Legitimate Keywords/Phrases:")
for word, weight in zip(feature_names[top_legit_idx], coefs[top_legit_idx]):
    print(f"  • {word:<25} (weight: {weight:+.2f})")

## 7. Interactive Sample Testing

In [ ]:
sample_emails = [
    "URGENT: Your account has been suspended due to unauthorized activity. Please verify your identity immediately at http://login-secure-auth.xyz to restore access within 24 hours.",
    "Hi team, attached is the revised meeting schedule for tomorrow afternoon. Let me know if you have any questions.",
    "Dear customer, we detected an invoice payment error. Click the attached link to update your billing details right away or your service will be terminated."
]

for email in sample_emails:
    prob = pipeline.predict_proba([email])[0][1]
    label = "PHISHING 🚨" if prob >= 0.50 else "LEGITIMATE ✅"
    print(f"[{label} - {prob*100:.1f}% risk] => {email[:75]}...")

## 8. Save Trained Model Pipeline

In [ ]:
save_path = os.path.join("backend", "ml", "saved_models", "nlp_pipeline.joblib")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
joblib.dump(pipeline, save_path)
print(f"Model artifact successfully exported to: {save_path}")